# IE7615 – Milestone 2 Demo
Loads our trained YOLOv8s weights (`best.pt`) and runs detection. No training happens here.

- **Single-object image:** `detect(path, single=True)`
- **Multi-object image:** `detect(path)`

Upload demo images with the folder icon on the left, or read them from Drive.

In [ ]:
!pip install -q ultralytics

Load the weights from our GitHub repo (works for anyone), or from Drive if you have the results folder.

In [ ]:
from ultralytics import YOLO
from PIL import Image
import matplotlib.pyplot as plt

import os

USE_GITHUB = True
GITHUB_WEIGHTS = "https://github.com/<username>/<repo>/raw/main/best.pt"   # fill in after uploading best.pt

if USE_GITHUB:
    os.system(f"wget -q -O best.pt {GITHUB_WEIGHTS}")
    WEIGHTS = "best.pt"
else:
    from google.colab import drive
    drive.mount('/content/drive')
    WEIGHTS = "/content/drive/MyDrive/IE7615_M2_results/best.pt"

model = YOLO(WEIGHTS)
print(len(model.names), "classes:", model.names[0], "...", model.names[len(model.names) - 1])

In [ ]:
def to_square(img):
    """Center-crop to a square and resize to 224x224, same as the dataset photos."""
    w, h = img.size
    s = min(w, h)
    img = img.crop(((w - s) // 2, (h - s) // 2, (w - s) // 2 + s, (h - s) // 2 + s))
    return img.resize((224, 224))


def detect(path, single=False, conf=0.4):
    img = Image.open(path).convert("RGB")

    if single:
        # Place the photo in the top-left cell of a 2x2-sized gray canvas.
        # On our 1,101 held-out test photos this gave 96.7% accuracy,
        # vs 82.7% when the photo was passed in directly.
        canvas = Image.new("RGB", (448, 448), (114, 114, 114))
        canvas.paste(to_square(img), (0, 0))
        img = canvas

    r = model.predict(img, imgsz=672, conf=conf, agnostic_nms=True, verbose=False)[0]

    if single and len(r.boxes) == 0:
        # nothing above the threshold: retry lower and keep the top guess
        r = model.predict(img, imgsz=672, conf=0.05, agnostic_nms=True, verbose=False)[0]

    boxes = sorted(r.boxes, key=lambda b: -float(b.conf))
    if single:
        boxes = boxes[:1]
        if boxes:
            b = boxes[0]
            print(f"Object ID: {model.names[int(b.cls)]}   confidence: {float(b.conf):.2f}")
        else:
            print("No object detected")
    else:
        print(f"{len(boxes)} objects found")
        print(f"{'ID':>8}  conf   box (x1, y1, x2, y2)")
        for b in sorted(boxes, key=lambda b: (round(float(b.xyxy[0][1]) / 100), float(b.xyxy[0][0]))):
            x1, y1, x2, y2 = map(int, b.xyxy[0].tolist())
            print(f"{model.names[int(b.cls)]:>8}  {float(b.conf):.2f}   ({x1}, {y1}, {x2}, {y2})")

    plt.figure(figsize=(7, 7))
    plt.imshow(r.plot()[:, :, ::-1])
    plt.axis("off")
    plt.show()

### a) Single object

In [ ]:
detect("/content/single.jpg", single=True)

### b) Multiple objects

In [ ]:
detect("/content/multi.jpg")